# 🏦 AML Transaction Monitoring Model
**Author:** Anushka Shinde | MS Finance, Boston University  
**Stage 4:** Excel Report Generation

---
So far:
- **Stage 1** → Built a dataset of 500 transactions
- **Stage 2** → Applied 7 rule-based AML red flags
- **Stage 3** → Scored every transaction 0–100 using ML

In Stage 4, we export everything into a **professional Excel report** with 4 sheets:

| Sheet | Contents |
|---|---|
| Executive Summary | KPI dashboard — key numbers at a glance |
| HIGH Risk Alerts | Only the high-risk transactions, for immediate review |
| All Transactions | Full dataset with risk scores and flags |
| Methodology | Explanation of rules and model — great for interviews |

> This is the deliverable you will show employers. A well-formatted Excel report demonstrates both your technical skills AND your ability to communicate findings — exactly what financial analysts and compliance roles require.

---
## Step 1: Re-run Stages 1, 2 & 3
This rebuilds your full dataset with red flags and risk scores.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

customer_types = ['Individual', 'Business', 'Shell Company', 'NGO']
countries = ['USA', 'UK', 'India', 'Cayman Islands', 'Panama',
             'Switzerland', 'Germany', 'UAE', 'Nigeria', 'Singapore']
high_risk_countries = ['Cayman Islands', 'Panama', 'Nigeria']
transaction_types = ['Wire Transfer', 'Cash Deposit', 'ATM Withdrawal',
                     'Online Transfer', 'Check', 'Crypto Exchange']
n = 500

df = pd.DataFrame({
    'Transaction_ID': [f'TXN{str(i).zfill(5)}' for i in range(1, n + 1)],
    'Customer_ID': [f'CUST{np.random.randint(1000, 2000)}' for _ in range(n)],
    'Customer_Type': np.random.choice(customer_types, n, p=[0.5, 0.3, 0.1, 0.1]),
    'Transaction_Amount': np.round(
        np.where(
            np.random.rand(n) > 0.95,
            np.random.uniform(9000, 9999, n),
            np.random.exponential(scale=3000, size=n).clip(100, 100000)
        ), 2),
    'Transaction_Type': np.random.choice(transaction_types, n),
    'Origin_Country': np.random.choice(countries, n,
        p=[0.3, 0.15, 0.15, 0.05, 0.05, 0.08, 0.1, 0.05, 0.04, 0.03]),
    'Destination_Country': np.random.choice(countries, n),
    'Num_Transactions_Last_30Days': np.random.randint(1, 50, n),
    'Avg_Transaction_Last_6Months': np.round(
        np.random.exponential(scale=2000, size=n).clip(100, 50000), 2),
    'Account_Age_Years': np.round(np.random.uniform(0.1, 20, n), 1),
    'Prior_SAR_Filed': np.random.choice([0, 1], n, p=[0.92, 0.08]),
})

def apply_red_flags(row):
    flags = []
    if 9000 <= row['Transaction_Amount'] <= 9999:
        flags.append('Structuring')
    if row['Origin_Country'] in high_risk_countries or \
       row['Destination_Country'] in high_risk_countries:
        flags.append('High-Risk Country')
    if row['Avg_Transaction_Last_6Months'] > 0:
        if row['Transaction_Amount'] / row['Avg_Transaction_Last_6Months'] > 5:
            flags.append('Unusual Amount vs History')
    if row['Customer_Type'] in ['Shell Company', 'NGO'] and \
       row['Transaction_Type'] == 'Wire Transfer':
        flags.append('High-Risk Entity Type')
    if row['Prior_SAR_Filed'] == 1:
        flags.append('Prior SAR on File')
    if row['Account_Age_Years'] < 1 and row['Transaction_Amount'] > 10000:
        flags.append('New Account High Value')
    if row['Num_Transactions_Last_30Days'] > 30:
        flags.append('Excessive Transaction Frequency')
    return '; '.join(flags) if flags else 'None'

df['Red_Flags'] = df.apply(apply_red_flags, axis=1)
df['Flag_Count'] = df['Red_Flags'].apply(
    lambda x: 0 if x == 'None' else len(x.split(';')))

features = ['Transaction_Amount', 'Num_Transactions_Last_30Days',
            'Avg_Transaction_Last_6Months', 'Account_Age_Years',
            'Prior_SAR_Filed', 'Flag_Count']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])
model = IsolationForest(contamination=0.08, random_state=42)
model.fit(X_scaled)
raw_scores = model.score_samples(X_scaled)
df['Risk_Score'] = ((raw_scores - raw_scores.max()) /
                    (raw_scores.min() - raw_scores.max()) * 100).round(1)
df['Risk_Tier'] = df['Risk_Score'].apply(
    lambda s: 'HIGH' if s >= 70 else ('MEDIUM' if s >= 40 else 'LOW'))
df['Alert'] = df['Risk_Tier'].apply(lambda x: 'YES' if x == 'HIGH' else 'NO')

print(f'✅ All stages complete — {len(df)} transactions scored and ready for export.')
print(f'   HIGH Risk : {(df["Risk_Tier"]=="HIGH").sum()}')
print(f'   MEDIUM    : {(df["Risk_Tier"]=="MEDIUM").sum()}')
print(f'   LOW       : {(df["Risk_Tier"]=="LOW").sum()}')

---
## Step 2: Install and Import openpyxl

**openpyxl** is the library that lets Python create and format Excel files.

- `Workbook` → creates a new Excel file
- `Font` → controls text style (bold, size, color)
- `PatternFill` → fills a cell with a background color
- `Alignment` → controls text alignment (left, center, wrap)
- `Border` / `Side` → adds borders around cells
- `get_column_letter` → converts column number to letter (e.g. 3 → C)

In [ ]:
!pip install openpyxl -q

import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print('✅ openpyxl loaded!')

---
## Step 3: Define Colors and Helper Functions

We define our color palette and two small helper functions upfront.
This keeps the code clean — instead of writing long style code every time,
we just call `fill('FF0000')` or `cell_font(bold=True)`.

Colors are defined as **hex codes** — the same format used in web design.
For example `FF4C4C` is red, `1F3864` is dark navy blue.

In [ ]:
# Color palette
COLOR_HEADER = '1F3864'   # Dark navy — used for main headers
COLOR_SUBHDR = '2E75B6'   # Medium blue — used for section headers
COLOR_HIGH   = 'FF4C4C'   # Red — HIGH risk
COLOR_MEDIUM = 'FFB84C'   # Orange — MEDIUM risk
COLOR_LOW    = '4CAF82'   # Green — LOW risk
COLOR_WHITE  = 'FFFFFF'

# Border style — thin grey lines around cells
thin = Side(style='thin', color='CCCCCC')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

# Helper: create a fill (background color)
def fill(hex_color):
    return PatternFill('solid', fgColor=hex_color)

# Helper: create a font style
def make_font(size=10, bold=False, color='000000'):
    return Font(name='Arial', size=size, bold=bold, color=color)

print('✅ Colors and helpers defined!')

---
## Step 4: Create the Workbook and Sheet 1 — Executive Summary

The Executive Summary is what a **senior compliance officer or manager** sees first.
It should answer: *How many alerts? What are the biggest risks? At a glance.*

We build:
1. A title banner
2. KPI boxes (key numbers)
3. A risk tier breakdown table
4. A red flag frequency table

In [ ]:
# Create a new Excel workbook
wb = Workbook()

# The workbook starts with one sheet — rename it
ws1 = wb.active
ws1.title = 'Executive Summary'

# ── TITLE BANNER ──────────────────────────────────────────
ws1.merge_cells('A1:H1')   # Merge cells A1 to H1 into one wide cell
ws1['A1'] = 'AML TRANSACTION MONITORING REPORT'
ws1['A1'].font = Font(name='Arial', bold=True, size=16, color=COLOR_WHITE)
ws1['A1'].fill = fill(COLOR_HEADER)
ws1['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws1.row_dimensions[1].height = 36

ws1.merge_cells('A2:H2')
ws1['A2'] = 'Prepared by: Anushka Shinde  |  MS Finance, Boston University  |  June 2026'
ws1['A2'].font = Font(name='Arial', italic=True, size=10, color='555555')
ws1['A2'].alignment = Alignment(horizontal='center')
ws1.row_dimensions[2].height = 20
ws1.row_dimensions[3].height = 10  # Spacer row

print('✅ Title banner created!')

In [ ]:
# ── KPI BOXES ─────────────────────────────────────────────
# Calculate the key numbers
total  = len(df)
high   = (df['Risk_Tier'] == 'HIGH').sum()
medium = (df['Risk_Tier'] == 'MEDIUM').sum()
low    = (df['Risk_Tier'] == 'LOW').sum()
struct = df['Red_Flags'].str.contains('Structuring').sum()
hrc    = df['Red_Flags'].str.contains('High-Risk Country').sum()
sar    = df['Red_Flags'].str.contains('Prior SAR').sum()
alert_rate = f"{round(high / total * 100, 1)}%"

kpi_labels = ['Total Transactions', 'HIGH Risk Alerts', 'MEDIUM Risk', 'LOW Risk',
              'Structuring Flags', 'High-Risk Country', 'Prior SAR Flags', 'Alert Rate']
kpi_values = [total, high, medium, low, struct, hrc, sar, alert_rate]
kpi_colors = [COLOR_SUBHDR, COLOR_HIGH, 'E87722', '2E8B57',
              '7030A0', 'C00000', '843C0C', COLOR_HEADER]

# Build each KPI box — label row (row 4) and value row (row 5)
for i, (label, value, color) in enumerate(zip(kpi_labels, kpi_values, kpi_colors)):
    col = i + 1
    ws1.column_dimensions[get_column_letter(col)].width = 18

    # Label cell
    c_lbl = ws1.cell(row=4, column=col, value=label)
    c_lbl.font = Font(name='Arial', bold=True, size=9, color=COLOR_WHITE)
    c_lbl.fill = fill(color)
    c_lbl.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    c_lbl.border = border
    ws1.row_dimensions[4].height = 32

    # Value cell
    c_val = ws1.cell(row=5, column=col, value=value)
    c_val.font = Font(name='Arial', bold=True, size=18, color=color)
    c_val.fill = fill('F7F9FC')
    c_val.alignment = Alignment(horizontal='center', vertical='center')
    c_val.border = border
    ws1.row_dimensions[5].height = 40

ws1.row_dimensions[6].height = 16  # Spacer
print('✅ KPI boxes created!')

In [ ]:
# ── RISK TIER BREAKDOWN TABLE ──────────────────────────────
ws1.merge_cells('A7:D7')
ws1['A7'] = 'RISK TIER BREAKDOWN'
ws1['A7'].font = Font(name='Arial', bold=True, size=11, color=COLOR_WHITE)
ws1['A7'].fill = fill(COLOR_SUBHDR)
ws1['A7'].alignment = Alignment(horizontal='center')

for j, h in enumerate(['Risk Tier', 'Count', '% of Total', 'Avg Risk Score'], start=1):
    c = ws1.cell(row=8, column=j, value=h)
    c.font = Font(name='Arial', bold=True, size=10, color=COLOR_WHITE)
    c.fill = fill(COLOR_HEADER)
    c.alignment = Alignment(horizontal='center')
    c.border = border

tier_color_map = {'HIGH': COLOR_HIGH, 'MEDIUM': COLOR_MEDIUM, 'LOW': COLOR_LOW}
for r_idx, tier in enumerate(['HIGH', 'MEDIUM', 'LOW'], start=9):
    count = (df['Risk_Tier'] == tier).sum()
    pct = f"{round(count / total * 100, 1)}%"
    avg = round(df[df['Risk_Tier'] == tier]['Risk_Score'].mean(), 1)
    for j, val in enumerate([tier, count, pct, avg], start=1):
        c = ws1.cell(row=r_idx, column=j, value=val)
        c.font = Font(name='Arial', size=10, bold=(j==1),
                      color=COLOR_WHITE if j==1 else '000000')
        c.fill = fill(tier_color_map[tier]) if j==1 else fill('F7F9FC')
        c.alignment = Alignment(horizontal='center')
        c.border = border

ws1.row_dimensions[13].height = 16  # Spacer
print('✅ Risk tier table created!')

In [ ]:
# ── RED FLAG FREQUENCY TABLE ───────────────────────────────
ws1.merge_cells('A14:D14')
ws1['A14'] = 'RED FLAG FREQUENCY'
ws1['A14'].font = Font(name='Arial', bold=True, size=11, color=COLOR_WHITE)
ws1['A14'].fill = fill(COLOR_SUBHDR)
ws1['A14'].alignment = Alignment(horizontal='center')

for j, h in enumerate(['Red Flag Type', 'Transactions Flagged', '% of Total', 'Avg Risk Score'], start=1):
    c = ws1.cell(row=15, column=j, value=h)
    c.font = Font(name='Arial', bold=True, size=10, color=COLOR_WHITE)
    c.fill = fill(COLOR_HEADER)
    c.alignment = Alignment(horizontal='center')
    c.border = border

flag_types = ['Structuring', 'High-Risk Country', 'Unusual Amount vs History',
              'High-Risk Entity Type', 'Prior SAR on File',
              'New Account High Value', 'Excessive Transaction Frequency']

for r_idx, flag in enumerate(flag_types, start=16):
    flagged = df[df['Red_Flags'].str.contains(flag, na=False)]
    count = len(flagged)
    pct = f"{round(count / total * 100, 1)}%"
    avg = round(flagged['Risk_Score'].mean(), 1) if count > 0 else 0
    bg = 'FFF2F2' if r_idx % 2 == 0 else 'FFFFFF'
    for j, val in enumerate([flag, count, pct, avg], start=1):
        c = ws1.cell(row=r_idx, column=j, value=val)
        c.font = Font(name='Arial', size=10, bold=(j==1))
        c.fill = fill(bg)
        c.alignment = Alignment(horizontal='center' if j > 1 else 'left', indent=1 if j==1 else 0)
        c.border = border

print('✅ Red flag table created!')

---
## Step 5: Sheet 2 — HIGH Risk Alerts

This sheet contains only the HIGH risk transactions.
In a real bank, this would go directly to a compliance analyst for review.

In [ ]:
ws2 = wb.create_sheet('HIGH Risk Alerts')

high_df = df[df['Risk_Tier'] == 'HIGH'].sort_values('Risk_Score', ascending=False).reset_index(drop=True)

ws2.merge_cells('A1:K1')
ws2['A1'] = f'HIGH RISK ALERTS — REQUIRES IMMEDIATE REVIEW ({len(high_df)} transactions)'
ws2['A1'].font = Font(name='Arial', bold=True, size=13, color=COLOR_WHITE)
ws2['A1'].fill = fill(COLOR_HIGH)
ws2['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws2.row_dimensions[1].height = 30

cols = ['Transaction_ID', 'Customer_ID', 'Customer_Type', 'Transaction_Amount',
        'Transaction_Type', 'Origin_Country', 'Destination_Country',
        'Red_Flags', 'Risk_Score', 'Risk_Tier', 'Alert']
col_widths = [14, 12, 16, 20, 18, 16, 20, 50, 12, 12, 8]

for j, (col, w) in enumerate(zip(cols, col_widths), start=1):
    ws2.column_dimensions[get_column_letter(j)].width = w
    c = ws2.cell(row=2, column=j, value=col.replace('_', ' ').title())
    c.font = Font(name='Arial', bold=True, size=10, color=COLOR_WHITE)
    c.fill = fill(COLOR_HEADER)
    c.alignment = Alignment(horizontal='center', wrap_text=True)
    c.border = border
ws2.row_dimensions[2].height = 28

for r_idx, row in enumerate(high_df.itertuples(), start=3):
    row_data = [row.Transaction_ID, row.Customer_ID, row.Customer_Type,
                row.Transaction_Amount, row.Transaction_Type, row.Origin_Country,
                row.Destination_Country, row.Red_Flags, row.Risk_Score,
                row.Risk_Tier, row.Alert]
    bg = 'FFF0F0' if r_idx % 2 == 0 else 'FFFFFF'
    for j, val in enumerate(row_data, start=1):
        c = ws2.cell(row=r_idx, column=j, value=val)
        c.font = Font(name='Arial', size=10,
                      bold=True if j==9 else False,
                      color=COLOR_HIGH if j==9 else '000000')
        c.fill = fill(bg)
        c.border = border
        if j == 4:
            c.number_format = '$#,##0.00'

print(f'✅ HIGH Risk Alerts sheet created — {len(high_df)} alerts!')

---
## Step 6: Sheet 3 — All Transactions

The full dataset with every transaction color-coded by risk tier.
Red rows = HIGH, orange = MEDIUM, green = LOW.

In [ ]:
ws3 = wb.create_sheet('All Transactions')

ws3.merge_cells('A1:K1')
ws3['A1'] = f'ALL TRANSACTIONS — RISK SCORED ({len(df)} records)'
ws3['A1'].font = Font(name='Arial', bold=True, size=13, color=COLOR_WHITE)
ws3['A1'].fill = fill(COLOR_SUBHDR)
ws3['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws3.row_dimensions[1].height = 30

for j, (col, w) in enumerate(zip(cols, col_widths), start=1):
    ws3.column_dimensions[get_column_letter(j)].width = w
    c = ws3.cell(row=2, column=j, value=col.replace('_', ' ').title())
    c.font = Font(name='Arial', bold=True, size=10, color=COLOR_WHITE)
    c.fill = fill(COLOR_HEADER)
    c.alignment = Alignment(horizontal='center', wrap_text=True)
    c.border = border
ws3.row_dimensions[2].height = 28

tier_bg = {'HIGH': 'FFF0F0', 'MEDIUM': 'FFF8EE', 'LOW': 'F0FFF4'}
tier_score_color = {'HIGH': COLOR_HIGH, 'MEDIUM': 'E87722', 'LOW': '2E8B57'}

for r_idx, row in enumerate(df.itertuples(), start=3):
    row_data = [row.Transaction_ID, row.Customer_ID, row.Customer_Type,
                row.Transaction_Amount, row.Transaction_Type, row.Origin_Country,
                row.Destination_Country, row.Red_Flags, row.Risk_Score,
                row.Risk_Tier, row.Alert]
    bg = tier_bg.get(row.Risk_Tier, 'FFFFFF')
    for j, val in enumerate(row_data, start=1):
        c = ws3.cell(row=r_idx, column=j, value=val)
        c.font = Font(name='Arial', size=10,
                      bold=True if j==9 else False,
                      color=tier_score_color.get(row.Risk_Tier, '000000') if j==9 else '000000')
        c.fill = fill(bg)
        c.border = border
        if j == 4:
            c.number_format = '$#,##0.00'

print('✅ All Transactions sheet created!')

---
## Step 7: Sheet 4 — Methodology

This sheet explains your rules and model in plain English.

Why include this?
- In real compliance work, every model needs **documentation** so auditors can review it
- In interviews, this sheet shows you understand the *why* behind every decision
- It turns your project from a code exercise into a **professional deliverable**

In [ ]:
ws4 = wb.create_sheet('Methodology & Notes')
ws4.column_dimensions['A'].width = 30
ws4.column_dimensions['B'].width = 70

ws4.merge_cells('A1:B1')
ws4['A1'] = 'METHODOLOGY & MODEL NOTES'
ws4['A1'].font = Font(name='Arial', bold=True, size=14, color=COLOR_WHITE)
ws4['A1'].fill = fill(COLOR_HEADER)
ws4['A1'].alignment = Alignment(horizontal='center', vertical='center')
ws4.row_dimensions[1].height = 34

sections = [
    ('PROJECT OVERVIEW', ''),
    ('Purpose', 'End-to-end AML transaction monitoring combining rule-based typologies with ML anomaly detection to risk-score 500 synthetic banking transactions.'),
    ('Dataset', '500 synthetic transactions simulating real banking data: wire transfers, cash deposits, crypto exchanges, and more across 10 countries.'),
    ('', ''),
    ('RULE-BASED FLAGS', ''),
    ('Structuring', 'Transactions $9,000–$9,999 flagged as potential structuring — deliberately staying below the $10,000 CTR threshold. Federal crime under 31 U.S.C. § 5324.'),
    ('High-Risk Country', 'Transactions involving FATF-listed high-risk jurisdictions: Cayman Islands, Panama, Nigeria.'),
    ('Unusual Amount vs History', 'Transaction amount > 5x the customer 6-month average — potential placement or layering indicator.'),
    ('High-Risk Entity Type', 'Shell companies or NGOs conducting wire transfers — common vehicles for layering illicit funds.'),
    ('Prior SAR on File', 'Customer has a prior Suspicious Activity Report — elevated inherent risk on all subsequent transactions.'),
    ('New Account High Value', 'Account opened < 1 year with transactions > $10,000 — new account risk indicator.'),
    ('Excessive Frequency', 'More than 30 transactions in 30 days — potential smurfing pattern.'),
    ('', ''),
    ('ML MODEL', ''),
    ('Algorithm', 'Isolation Forest — unsupervised anomaly detection. Randomly partitions the feature space; anomalies are isolated in fewer splits than normal transactions.'),
    ('Why Unsupervised?', 'Real AML data rarely has confirmed fraud labels. Unsupervised ML learns what normal looks like and flags deviations — no labeled data required.'),
    ('Features Used', 'Transaction Amount, # Transactions (30 days), Avg Transaction (6M), Account Age, Prior SAR, Red Flag Count'),
    ('Normalization', 'StandardScaler applied before training — ensures no single feature dominates due to scale differences.'),
    ('Contamination', '8% — assumed proportion of anomalous transactions, consistent with industry AML alert rate benchmarks.'),
    ('Risk Scoring', 'Raw Isolation Forest scores normalized to 0–100 scale. Higher = more anomalous = higher risk.'),
    ('', ''),
    ('RISK TIERS', ''),
    ('HIGH  (Score >= 70)', 'Escalate immediately — review for SAR filing with FinCEN / FIU-IND.'),
    ('MEDIUM (Score 40-69)', 'Enhanced Due Diligence (EDD) required — ongoing monitoring.'),
    ('LOW   (Score < 40)', 'Standard monitoring — no immediate action required.'),
    ('', ''),
    ('LIMITATIONS', ''),
    ('Synthetic Data', 'Real implementation requires live transaction feeds and full customer KYC data.'),
    ('Model Improvement', 'Supervised models (XGBoost, logistic regression) would improve precision with labeled SAR data.'),
    ('Threshold Calibration', 'Risk tier cutoffs (70/40) are illustrative. Production models are calibrated against historical SAR outcomes.'),
]

for r_idx, (key, val) in enumerate(sections, start=2):
    c_key = ws4.cell(row=r_idx, column=1, value=key)
    c_val = ws4.cell(row=r_idx, column=2, value=val)
    if val == '' and key != '':
        ws4.merge_cells(f'A{r_idx}:B{r_idx}')
        c_key.font = Font(name='Arial', bold=True, size=11, color=COLOR_WHITE)
        c_key.fill = fill(COLOR_SUBHDR)
        c_key.alignment = Alignment(horizontal='left', indent=1)
        ws4.row_dimensions[r_idx].height = 24
    elif key == '':
        ws4.row_dimensions[r_idx].height = 8
    else:
        bg = 'F0F4FA' if r_idx % 2 == 0 else 'FFFFFF'
        c_key.font = Font(name='Arial', bold=True, size=10)
        c_val.font = Font(name='Arial', size=10)
        c_val.alignment = Alignment(wrap_text=True)
        for c in [c_key, c_val]:
            c.fill = fill(bg)
            c.border = border
        ws4.row_dimensions[r_idx].height = 30

print('✅ Methodology sheet created!')

---
## Step 8: Save the File and Download It

We save the workbook to a file, then use Google Colab's `files.download()` to download it to your computer.

In [ ]:
from google.colab import files

output_path = 'AML_Transaction_Monitoring_Report.xlsx'
wb.save(output_path)

print(f'✅ Report saved as: {output_path}')
print('\n📥 Downloading now...')

files.download(output_path)

print('\n🎉 Done! Check your downloads folder.')

---
## ✅ Stage 4 Complete — Your Project is Done!

You now have a fully built, professionally formatted AML Transaction Monitoring report.

### What you built across all 4 stages:

| Stage | What you did | Real-world equivalent |
|---|---|---|
| Stage 1 | Generated 500 transactions | Bank transaction database |
| Stage 2 | Applied 7 AML typology rules | Compliance rule engine (Actimize, Mantas) |
| Stage 3 | ML anomaly scoring 0–100 | AI risk scoring layer |
| Stage 4 | Professional Excel report | Deliverable to senior compliance officer |

---
### How to describe this project in interviews:

> *"I built an end-to-end AML transaction monitoring system in Python. It applies seven rule-based typologies — including structuring detection, high-risk country screening, and smurfing indicators — combined with an Isolation Forest ML model that scores each transaction from 0 to 100 for anomaly risk. The output is a formatted Excel report with an executive summary dashboard, a HIGH risk alert sheet for immediate compliance review, and a full methodology page. The project is directly informed by my experience at Deloitte working on forensic and financial crime investigations."*

---
**Next → Stage 5: Upload to GitHub + Final Resume Line**  
We will put this project on GitHub so you can share it with employers — and write the perfect resume bullet point.